In [4]:
import os
import glob
import pandas as pd
import numpy as np
import sqlite3  # Built-in to Python, no installation needed

print("Pandas version:", pd.__version__)

Pandas version: 3.0.1


In [5]:
# Path to your data folder
data_dir = r"D:\ExcelR\Projects\Live Climate\Data"

# Find all CSV files in the folder
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
print(f"Found {len(csv_files)} files:\n")

dfs = []
for file in csv_files:
    filename = os.path.basename(file)
    df_temp = pd.read_csv(file, skiprows=9)
    df_temp["SOURCE_FILE"] = filename
    dfs.append(df_temp)
    print(f"✓ Loaded: {filename} -> Shape: {df_temp.shape}")

Found 4 files:

✓ Loaded: POWER_Regional_Daily_20240101_20240107_006d00N_016d00N_073d00E_083d00E_LST.csv -> Shape: (2352, 7)
✓ Loaded: POWER_Regional_Daily_20240101_20240107_016d00N_026d00N_073d00E_083d00E_LST.csv -> Shape: (2352, 7)
✓ Loaded: POWER_Regional_Daily_20240101_20240107_020d00N_030d00N_083d00E_093d00E_LST.csv -> Shape: (2352, 7)
✓ Loaded: POWER_Regional_Daily_20240101_20240107_026d00N_036d00N_068d00E_078d00E_LST.csv -> Shape: (2352, 7)


## Combine All Files into One Table


In [8]:
# Combine all 4 tables together
df = pd.concat(dfs, ignore_index=True)

# See the total rows and preview first 5 rows
print("Total rows:", len(df))
df.head()

Total rows: 9408


,LAT,LON,YEAR,MO,DY,T2M,SOURCE_FILE
0,6.0,73.125,2024,1,1,27.42,POWER_Regional_Daily_20240101_20240107_006d00N...
1,6.0,73.750,2024,1,1,27.28,POWER_Regional_Daily_20240101_20240107_006d00N...
2,6.0,74.375,2024,1,1,27.18,POWER_Regional_Daily_20240101_20240107_006d00N...
3,6.0,75.000,2024,1,1,27.22,POWER_Regional_Daily_20240101_20240107_006d00N...
4,6.0,75.625,2024,1,1,27.36,POWER_Regional_Daily_20240101_20240107_006d00N...


## Remove Duplicate Rows


In [9]:
# Drop duplicate rows based on coordinates and date
df = df.drop_duplicates(subset=['LAT', 'LON', 'YEAR', 'MO', 'DY'])

# Reset index to keep it clean
df = df.reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 9240


## Check for Missing Values & Data Types

In [10]:
# Check for null values in each column
print(df.isnull().sum())

# basic statistics
df.describe()

LAT            0
LON            0
YEAR           0
MO             0
DY             0
T2M            0
SOURCE_FILE    0
dtype: int64


,LAT,LON,YEAR,MO,DY,T2M
count,9240.000000,9240.000000,9240.0,9240.0,9240.000000,9240.000000
mean,22.048485,79.100379,2024.0,1.0,4.000000,15.899952
std,7.922631,6.201975,0.0,0.0,2.000108,10.851385
min,6.000000,68.125000,2024.0,1.0,1.000000,-26.970000
25%,16.000000,74.375000,2024.0,1.0,2.000000,12.650000
50%,23.000000,78.125000,2024.0,1.0,4.000000,17.690000
75%,28.000000,83.125000,2024.0,1.0,6.000000,23.360000
max,36.000000,92.500000,2024.0,1.0,7.000000,29.080000


## Create a Simple Date Column

In [11]:
# Create a proper Date column (e.g., 2024-01-01)
df['Date'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + 
                            df['MO'].astype(str) + '-' + 
                            df['DY'].astype(str))

# View the new Date column
df[['Date', 'LAT', 'LON', 'T2M']].head()

,Date,LAT,LON,T2M
0,2024-01-01,6.0,73.125,27.42
1,2024-01-01,6.0,73.750,27.28
2,2024-01-01,6.0,74.375,27.18
3,2024-01-01,6.0,75.000,27.22
4,2024-01-01,6.0,75.625,27.36


## Add Simple Temperature Features for Power BI

In [12]:
# 1. Heating Degree Days (HDD): High heating need if temperature is below 18°C
df['Heating_Demand'] = (18 - df['T2M']).clip(lower=0).round(1)

# 2. Cooling Degree Days (CDD): High AC/Cooling need if temperature is above 18°C
df['Cooling_Demand'] = (df['T2M'] - 18).clip(lower=0).round(1)

# 3. Frost Risk Alert: 1 if temperature is 4°C or below (dangerous for crops), else 0
df['Frost_Alert'] = (df['T2M'] <= 4).astype(int)

# Check the new columns
df[['T2M', 'Heating_Demand', 'Cooling_Demand', 'Frost_Alert']].head()

,T2M,Heating_Demand,Cooling_Demand,Frost_Alert
0,27.42,0.0,9.4,0
1,27.28,0.0,9.3,0
2,27.18,0.0,9.2,0
3,27.22,0.0,9.2,0
4,27.36,0.0,9.4,0


## Add Simple Region Names

In [13]:
# Simple function to assign regions based on latitude
def get_region(lat):
    if lat >= 30:
        return "North / Himalayas"
    elif lat >= 22:
        return "North & Central Plains"
    elif lat >= 15:
        return "Central & Western"
    else:
        return "South & Coastal"

# Apply to create a Region column
df['Region'] = df['LAT'].apply(get_region)

# See how many rows in each region
print(df['Region'].value_counts())

Region
North & Central Plains    3640
South & Coastal           2016
Central & Western         2016
North / Himalayas         1568
Name: count, dtype: int64


## Save the Cleaned File for Power BI & SQL

In [14]:
# Save clean file in the same Data folder
output_file = r"D:\ExcelR\Projects\Live Climate\Data\climate_clean_data.csv"
df.to_csv(output_file, index=False)

print("Saved successfully to:", output_file)

Saved successfully to: D:\ExcelR\Projects\Live Climate\Data\climate_clean_data.csv
